# Phase-3 Provenance Audit — seed46 clean single replay

**目的**：在当前 main HEAD 的代码上重跑 `phnode_full clean seed46` 一个 run，判断 catalog 时代的 fragility（epoch 26 起 ODE solver 全 fail，best_loss=0.27，60s rollout 47 m）在当前代码上是否仍可复现。

**为什么只跑 seed46**：Phase 2 同口径对齐显示 catalog vs cleanrun v1 的 15.7× gap 由 seed46 (103×) + seed42 (7.3×) 完全驱动；seed46 是训练发散的 catastrophic 失效，是 fragility 最强信号。单跑 seed46 < 5 min，能立即判定 fragility 是否自愈。

**云端路径约定**：参照 `notebook/phase1a_oc_v4lite_formal_workflow.ipynb`：

- Colab Drive 上工作目录是 `/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7`（路径名沿用 cleanrun v1 约定）
- 该目录的代码内容应该是**本地 g3_5_5 仓库 `provenance-audit-phnode_full` 分支 HEAD** 的手动同步镜像
- Colab 上不是 git 仓库，所以本 notebook 不调用 `git` 命令；commit hash 由你在 Cell 3 手动粘贴注入

**输出**：`checkpoints/audit_phase3_seed46_clean_<RUN_TAG>/` 一份完整 run，结尾打包为 `phase3_seed46_replay_<RUN_TAG>.tar.gz` **落到 Drive**，你手动下载即可。

**判读**（见 Cell 9）：
- 若 training.log 出现 "no successful training batches"，或 best_epoch < 30，或 60s clean pos_err_median > 10 m → fragility **仍可复现**，本地走 Phase 3 Setup B（git bisect）。
- 若 best_epoch ≈ 250 且 60s clean pos_err_median < 1.5 m → fragility **已自愈**，转 cfg/code swap 归因。


## 0. 环境检查

In [1]:
!lscpu | head -20

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  12
On-line CPU(s) list:                     0-11
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   85
Thread(s) per core:                      2
Core(s) per socket:                      6
Socket(s):                               1
Stepping:                                7
BogoMIPS:                                4400.33
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_fre

In [2]:
!nvidia-smi

Tue May 12 12:34:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"CUDA version: {torch.version.cuda}")

PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002
CUDA version: 12.8


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Project configuration

云端目录约定见顶部说明。如果你的镜像在别处，改 `AUV_PROJECT_DIR` 环境变量即可。


In [5]:
import os
from pathlib import Path

PROJECT_DIR = Path(os.environ.get(
    "AUV_PROJECT_DIR",
    "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7",
))
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
%cd $PROJECT_DIR

/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7


In [6]:
%pip install -q torchdiffeq pandas

## 2. 标记 Drive 上的 audit 输出目录

Audit 结果会先写到本工作目录的 `checkpoints/<RUN_TAG>/`（这本身就在 Drive 上），跑完后再额外打 tarball 拷贝到 `AUDIT_DROP_DIR`，方便你识别要下载哪个文件。


In [9]:
import os
from datetime import datetime
from pathlib import Path

# Audit 标识
os.environ["AUDIT_RUN_TAG"] = f"audit_phase3_seed46_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.environ["AUDIT_DATASET"] = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
os.environ["AUDIT_NOISE_REFERENCE"] = "remus100_dr"
os.environ["AUDIT_SUITE_NAME"] = os.environ["AUDIT_RUN_TAG"]
os.environ["AUDIT_DEVICE"] = "cuda"

# ⚠️ 在本地 g3_5_5 仓库跑 `git rev-parse HEAD`，把结果粘贴到下面这一行：
os.environ["AUDIT_LOCAL_GIT_COMMIT"] = "7643dc9"
# 同样可选：你本地分支名
os.environ["AUDIT_LOCAL_GIT_BRANCH"] = "provenance-audit-phnode_full"

# 产物 drop 目录（你本地手动下载用）
AUDIT_DROP_DIR = Path("checkpoints") / "_phase3_audit_drops"
AUDIT_DROP_DIR.mkdir(parents=True, exist_ok=True)
os.environ["AUDIT_DROP_DIR"] = str(AUDIT_DROP_DIR)

print("AUDIT_RUN_TAG          =", os.environ["AUDIT_RUN_TAG"])
print("AUDIT_DATASET          =", os.environ["AUDIT_DATASET"])
print("AUDIT_NOISE_REFERENCE  =", os.environ["AUDIT_NOISE_REFERENCE"])
print("AUDIT_SUITE_NAME       =", os.environ["AUDIT_SUITE_NAME"])
print("AUDIT_DEVICE           =", os.environ["AUDIT_DEVICE"])
print("AUDIT_LOCAL_GIT_COMMIT =", os.environ["AUDIT_LOCAL_GIT_COMMIT"])
print("AUDIT_LOCAL_GIT_BRANCH =", os.environ["AUDIT_LOCAL_GIT_BRANCH"])
print("AUDIT_DROP_DIR         =", os.environ["AUDIT_DROP_DIR"])

if "FILL_ME" in os.environ["AUDIT_LOCAL_GIT_COMMIT"]:
    print("\n⚠️  请先回到本地仓库跑 `git rev-parse HEAD`，把 commit hash 填到本 cell 再继续。")
    print("    （不填也能跑，但产物里就缺这条 provenance 证据。）")

AUDIT_RUN_TAG          = audit_phase3_seed46_clean_20260512_095957
AUDIT_DATASET          = data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
AUDIT_NOISE_REFERENCE  = remus100_dr
AUDIT_SUITE_NAME       = audit_phase3_seed46_clean_20260512_095957
AUDIT_DEVICE           = cuda
AUDIT_LOCAL_GIT_COMMIT = 7643dc9
AUDIT_LOCAL_GIT_BRANCH = provenance-audit-phnode_full
AUDIT_DROP_DIR         = checkpoints/_phase3_audit_drops


## 3. Train: phnode_full × seed46 × clean (~3–5 min Colab L4/T4)

调用同 cleanrun v1 一致的 `train_all_models_noise_profile.sh`，只把 models 缩到 `phnode_full`、seeds 缩到 `46`。所有超参经由 wrapper → `train_auv_hamnode.py` → `DATASET_TRAINING_DEFAULTS["oc"]` 默认值（batch=4096 / epochs=300 / lr=6e-3 / warmup=400 / total=5000），与 cleanrun v1 完全一致。

`--block-eval-noise-profiles` / `--heldout-eval-noise-profiles` 限定为 `clean`，省掉无关 eval profile。


In [ ]:
!bash scripts/train_all_models_noise_profile.sh \
  --profile oc \
  --models "phnode_full" \
  --dataset "${AUDIT_DATASET}" \
  --seeds "46" \
  --suite-name "${AUDIT_SUITE_NAME}" \
  --noise-profile clean \
  --noise-protocol auto \
  --noise-reference "${AUDIT_NOISE_REFERENCE}" \
  --block-eval-noise-profiles clean \
  --heldout-eval-noise-profiles clean \
  --device "${AUDIT_DEVICE}" 2>&1 | tee "checkpoints/${AUDIT_SUITE_NAME}_train_stdout.log"

Training all models with profile-based noisy IC.
Dataset: data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Group: all
Explicit models: phnode_full
Seeds: 46
Noise config: protocol=auto, profile=clean, scale=1.0, warmup=20, ramp=80, mix_ratio=0.5
Auto-eval profiles: block=[clean] heldout=[clean]
Suite directory: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/audit_phase3_seed46_clean_20260512_095957
Dataset: data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Models (1): phnode_full
Seeds: 46
[train] main_phnode_full_seed46
Loading dataset...
Loaded: 83850 train, 21000 test, t_eval=[0.   0.05 0.1  0.15 0.2 ]
Dataset split: trajectory | train trajectories: 559 | test trajectories: 140
Dataset ID: d0be9434
Dataset path: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_5/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl | velocity convention: body_total | state dim: 27
Position std: [0.1508, 0.1400, 0.0370]
Velocity std: u=0.4722, v=0.0898, w=0.0718, p=0.0248, q=0.0906, r=0.052

## 4. 训练 log 关键诊断

三条信号要看：
1. 有没有 "no successful training batches" 行 — 训练发散标志（catalog A46 有 275 行）
2. best epoch / best loss — catalog A46 是 epoch 21 / loss 0.27；cleanrun v1 是 epoch 250 / loss 0.004
3. 完整最后 20 行 — 看 heldout 评估是否完成、有无异常 message


In [10]:
import subprocess
from pathlib import Path

suite_dir = Path("checkpoints") / os.environ["AUDIT_SUITE_NAME"]
run_dirs = sorted(suite_dir.glob("*phnode_full*seed46*"))
assert run_dirs, f"No phnode_full seed46 run found under {suite_dir}"
RUN_DIR = run_dirs[0]
print("RUN_DIR =", RUN_DIR)

log_path = RUN_DIR / "training.log"
print("\n=== diagnostic 1: count of 'no successful training batches' ===")
nofail = subprocess.run(
    ["grep", "-c", "no successful training batches", str(log_path)],
    capture_output=True, text=True,
)
print("count =", nofail.stdout.strip() or "0")

print("\n=== diagnostic 2: best epoch / best loss (final summary line) ===")
subprocess.run(["grep", "-E", "Best validation score|Done\\.", str(log_path)])

print("\n=== diagnostic 3: last 20 lines ===")
subprocess.run(["tail", "-20", str(log_path)])

print("\n=== diagnostic 4: any WARNING / ERROR / NaN / Inf ===")
subprocess.run(["grep", "-cE", "WARNING|ERROR|NaN|Inf|nan", str(log_path)])

RUN_DIR = checkpoints/audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46

=== diagnostic 1: count of 'no successful training batches' ===
count = 0

=== diagnostic 2: best epoch / best loss (final summary line) ===

=== diagnostic 3: last 20 lines ===

=== diagnostic 4: any WARNING / ERROR / NaN / Inf ===


CompletedProcess(args=['grep', '-cE', 'WARNING|ERROR|NaN|Inf|nan', 'checkpoints/audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/training.log'], returncode=1)

## 5. 60s clean rollout — 与 cleanrun v1 / catalog 同协议

调用 `eval_all_models_noise_profile.sh` 在该 suite 上跑 rollout benchmark。会按 wrapper 自动选 eval profile。


In [ ]:
!bash scripts/eval_all_models_noise_profile.sh --suite-dir "checkpoints/${AUDIT_SUITE_NAME}" 2>&1 | tee "checkpoints/${AUDIT_SUITE_NAME}_eval_stdout.log"

[stage=eval]
Suite directory: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/audit_phase3_seed46_clean_20260512_095957
Manifest: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/audit_phase3_seed46_clean_20260512_095957/runs.tsv
Mode: resampled
[eval] main_phnode_full_seed46
Starting rollout benchmark | checkpoint=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/best_model.pt | device=cuda | mode=resampled | noise_profile=clean | noise_protocol=clean | noise_budget=protocol=clean | profile=clean | ref=remus100_dr | epoch_scale=0.000 | mix_ratio=0.000 | nu_r_lin_floor[u=0.0000, v=0.0000, w=0.0000] | nu_r_lin_ratio[u=0.0000, v=0.0000, w=0.0000] | nu_r_ang[p=0.0000, q=0.0000, r=0.0000] | rot[roll=0.0000, pitch=0.0000, yaw=0.0000] | scenarios=PRBS,CHIRP,OU | trajectories=90 | max_horizon=60.0s | output_dir=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/c

## 6. 读出 rollout summary（60s × clean × pos_err）

In [11]:
import json
from pathlib import Path

rb_dirs = sorted((RUN_DIR / "rollout_benchmark").glob("*"))
rb_dirs = [d for d in rb_dirs if d.is_dir() and not d.name.startswith("_")]
print("rollout_benchmark subdirs:")
for d in rb_dirs:
    print(" ", d.name)

# 找 clean profile 的 summary.json
found = []
for d in rb_dirs:
    for sj in d.rglob("summary.json"):
        if "clean" in str(sj).split(d.name, 1)[-1]:
            found.append(sj)

print("\nclean summary.json candidates:")
for sj in found:
    print(" ", sj)

if found:
    with open(found[0]) as f:
        summary = json.load(f)
    print("\n=== 60s × clean × final_position_error ===")
    try:
        bucket = summary["overall"]["60.0"]["metrics"]["final_position_error"]
        print(json.dumps(bucket, indent=2))
    except Exception as e:
        print("summary schema differs; full dump (first 1500 chars):")
        print(json.dumps(summary, indent=2)[:1500])

rollout_benchmark subdirs:
  resampled_traj30_seed42_20260512_102603

clean summary.json candidates:
  checkpoints/audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/rollout_benchmark/resampled_traj30_seed42_20260512_102603/clean/summary.json

=== 60s × clean × final_position_error ===
{
  "count": 90,
  "mean": 0.6140771682770952,
  "median": 0.45575005972892096,
  "p90": 1.2999654655782376,
  "p95": 1.5137794424281459,
  "max": 1.928696910172036
}


## 7. 写 audit meta + 打包 + 落到 Drive

In [12]:
import subprocess, sys, os
from pathlib import Path
import torch

audit_meta_dir = Path("checkpoints") / os.environ["AUDIT_SUITE_NAME"] / "_audit_meta"
audit_meta_dir.mkdir(parents=True, exist_ok=True)

with open(audit_meta_dir / "code_revision.txt", "w") as f:
    f.write(f"local_git_commit: {os.environ.get('AUDIT_LOCAL_GIT_COMMIT', 'unknown')}\n")
    f.write(f"local_git_branch: {os.environ.get('AUDIT_LOCAL_GIT_BRANCH', 'unknown')}\n")
    f.write(f"colab_project_dir: {os.environ.get('AUV_PROJECT_DIR', '')}\n")
    f.write("note: Colab 镜像不是 git 仓库；以上 commit 由本地人工填入。\n")

with open(audit_meta_dir / "environment.txt", "w") as f:
    f.write(f"python: {sys.version}\n")
    f.write(f"torch:  {torch.__version__}\n")
    f.write(f"cuda:   {torch.version.cuda}\n")
    f.write(f"cudnn:  {torch.backends.cudnn.version()}\n")
    r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    f.write("gpu:\n" + r.stdout + "\n")

print("audit meta files:")
for p in audit_meta_dir.iterdir():
    print(" ", p)
print("---- code_revision.txt ----")
print((audit_meta_dir / "code_revision.txt").read_text())

# 打 tarball 落到 Drive，方便你手动下载
suite_dir = Path("checkpoints") / os.environ["AUDIT_SUITE_NAME"]
drop_dir = Path(os.environ["AUDIT_DROP_DIR"])
tarball = drop_dir / f"{os.environ['AUDIT_SUITE_NAME']}.tar.gz"

print("\nCreating tarball...")
result = subprocess.run(
    ["tar", "-czf", str(tarball), "-C", "checkpoints", os.environ["AUDIT_SUITE_NAME"]],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"tar warning/error (code {result.returncode}):")
    print(result.stderr.strip())

if tarball.exists():
    size = subprocess.run(["du", "-h", str(tarball)], capture_output=True, text=True).stdout.strip()
    print(f"\nwrote tarball to Drive: {tarball}")
    print("size:", size)
    print("\n→ 在 Drive 上找到该文件，手动下载到本地。")
else:
    print("\nFailed to create tarball.")

audit meta files:
  checkpoints/audit_phase3_seed46_clean_20260512_095957/_audit_meta/code_revision.txt
  checkpoints/audit_phase3_seed46_clean_20260512_095957/_audit_meta/environment.txt
---- code_revision.txt ----
local_git_commit: 7643dc9
local_git_branch: provenance-audit-phnode_full
colab_project_dir: 
note: Colab 镜像不是 git 仓库；以上 commit 由本地人工填入。


Creating tarball...
tar warning/error (code 1):
tar: audit_phase3_seed46_clean_20260512_095957/suite_config.txt: file changed as we read it
tar: audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/config.json: file changed as we read it
tar: audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/checkpoint_50.pt: file changed as we read it
tar: audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/checkpoint_100.pt: file changed as we read it
tar: audit_phase3_seed46_clean_20260512_095957/main_phnode_full_seed46/checkpoint_150.pt: file changed as we read it
tar: audit_phase3_seed46_clean_20260512_09595

## 8. 决策矩阵（本地分析时参照）

下载到本地后放进 `analysis/provenance_audit/phase3_retrain/`：

```bash
mkdir -p analysis/provenance_audit/phase3_retrain
tar -xzf audit_phase3_seed46_clean_<RUN_TAG>.tar.gz -C analysis/provenance_audit/phase3_retrain
```

判读规则：

| 信号 | 触发条件 | 解读 | 下一步 |
| --- | --- | --- | --- |
| **A** | training.log 含 "no successful training batches" ≥ 1 行 | fragility 在当前 main 仍可复现 | 本地 git bisect on g3_5_5 仓库，在 cleanrun v1 时代 commit ↔ 当前 HEAD 之间找修复发散的 commit |
| **B** | best_epoch ≥ 200 且 60s clean pos_err_median < 1.5 m | fragility 已自愈 | 单变量 swap：dataset / 各超参 / code path，定位自愈来源 |
| **C** | best_epoch ∈ [30, 200) 或 60s pos_err_median ∈ [1.5, 10) m | 部分发散 / 半自愈 | 复跑 1 次确认是否随 CUDA 非确定性波动；再决定 |
| **D** | invocation 报错 / eval 未完成 / 缺关键文件 | 调用链有问题 | 确认 `AUV_PROJECT_DIR` 镜像内容是否真的对应 `AUDIT_LOCAL_GIT_COMMIT`；检查 dataset 路径是否存在 |

**记得**：Cell 3 中 `AUDIT_LOCAL_GIT_COMMIT` 是 provenance 唯一手段，最好在跑训练前已经填好。
